In [7]:
from dotenv import load_dotenv
load_dotenv()

True

# 创建工具
- @tool(description = ...)修饰函数
- """说明"""

In [8]:
from langchain.tools import tool
@tool(description="输入地名的字符串，返回该地区的天气信息字符串")
def get_weather(location: str)->str:
    """
    Args:
        location:str  __the city name or coordinates

    Result:
        str the weather

    """
    
    return f"Current weather in {location} is sunny"

# Pydantic Model描述参数

In [9]:
from pydantic import BaseModel,Field
from typing import Literal
class WeatherInput(BaseModel):
    location:str = Field(description="City name or coordinates")
    units: Literal["celsius","fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast:bool =Field(
        default=False,
        description="Include 5-day forecast"
    )

In [10]:
@tool(args_schema=WeatherInput)
def Get_Weather(location:str,units:str = "celsius",include_forecats:bool=False):
    """Get current weather and optional forecast"""
    temp = 22 if units == "celisus" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecats:
        result += "\nNext 5 dayts: Sunny"
    return result

# 创建模型

In [11]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
model1 = init_chat_model(
    "deepseek-chat"
)

agent = create_agent(
    model1,
    tools = [
        get_weather
    ]
)

# 请求

In [12]:
from langchain_core.messages.human import HumanMessage

response = agent.invoke(
    {"messages":[
        HumanMessage("杭州天气如何")
    ]}
)
for res in response["messages"]:
    res.pretty_print()
    print(res.model_dump_json(indent=2))

================================ Human Message =================================

杭州天气如何
{
  "content": "杭州天气如何",
  "additional_kwargs": {},
  "response_metadata": {},
  "type": "human",
  "name": null,
  "id": "eae894ed-7ef1-4281-8dfb-e019315ded31"
}
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_fSvMwQak2sFwM2H915T52234)
 Call ID: call_00_fSvMwQak2sFwM2H915T52234
  Args:
    location: 杭州
{
  "content": "",
  "additional_kwargs": {
    "refusal": null
  },
  "response_metadata": {
    "token_usage": {
      "completion_tokens": 44,
      "prompt_tokens": 282,
      "total_tokens": 326,
      "completion_tokens_details": null,
      "prompt_tokens_details": {
        "audio_tokens": null,
        "cache_write_tokens": null,
        "cached_tokens": 256
      },
      "prompt_cache_hit_tokens": 256,
      "prompt_cache_miss_tokens": 26
    },
    "model_provider": "deepseek",
    "model_name": "deepseek-v4-flash",
    